# DC-EGM for discrete-continuous choice

Notes at <https://dse.iskh.me/dcegm>.

## Discrete and continuous choice together

## The retirement model

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

class retirement():
    '''Consumption-savings model with a discrete retirement choice (Iskhakov et al. 2017)'''

    def __init__(self, beta=0.95, R=1.05, y=1.0, delta=0.5, T=10, sigma=0.0, sigma_y=0.0,
                 Mbar=20, ngrid=500, nquad=5):
        self.beta, self.R, self.y, self.delta = beta, R, y, delta   # discount, return, wage, disutility of work
        self.T, self.sigma, self.sigma_y = T, sigma, sigma_y  # periods, taste shock scale, income shock std
        self.Mbar, self.ngrid, self.nquad = Mbar, ngrid, nquad
        self.eps = np.finfo(float).eps                        # smallest positive float
        self.A = np.linspace(0, Mbar, ngrid)                  # grid on savings, ascending, starts at zero
        x, w = np.polynomial.hermite.hermgauss(nquad)         # Gauss-Hermite nodes and weights
        self.quadp = y * np.exp(np.sqrt(2) * sigma_y * x - sigma_y**2 / 2)   # income nodes, mean y
        self.quadw = w / np.sqrt(np.pi)                       # weights, sum to one

    def u(self, c, d):
        return np.log(c) - self.delta * d                     # utility, d = 1 working, d = 0 retired

    def mu(self, c):
        return 1 / c                                          # marginal utility

    def imu(self, x):
        return 1 / x                                          # inverse marginal utility

    def next_wealth(self, A, d, y):
        return self.R * A + d * y                             # budget: income only while working

def interp(x, xp, fp):
    '''Linear interpolation, extrapolated linearly beyond the last grid point'''
    slope = (fp[-1] - fp[-2]) / (xp[-1] - xp[-2])
    return np.where(x > xp[-1], fp[-1] + slope * (x - xp[-1]), np.interp(x, xp, fp))

## Choice-specific value functions

### The credit constrained region

In [ ]:
def terminal(m, d):
    '''Last period: everything is consumed, no endogenous points, the whole grid is the constrained region'''
    pass                                                  # your code here

def policy(m, s, M):
    '''Consumption at wealth M from the solution s of one period and choice: c = M below the first endogenous point'''
    pass                                                  # your code here

def value(m, s, d, M):
    '''Choice-specific value at wealth M: analytic below the first endogenous point, interpolated above'''
    pass                                                  # your code here

### The discrete choice

In [ ]:
def logsum(m, vW, vR):
    '''Expected value over the taste shocks: the maximum without shocks, the smoothed logsum with'''
    pass                                                  # your code here

def choice_prob(m, vW, vR):
    '''Probability to keep working: an indicator without taste shocks, logit with'''
    pass                                                  # your code here

### The EGM step for one discrete choice

In [ ]:
def egm_step(m, d, sol1):
    '''EGM step for choice d (1 work, 0 retire) given the solution sol1 = [retire, work] of next period'''
    pass                                                  # your code here

In [ ]:
m = retirement()
sol = [[terminal(m, d) for d in (0, 1)]]                      # last period, index T-1
for t in range(2):
    sol.insert(0, [dict(zip(('M', 'c', 'v', 'ev0'), egm_step(m, d, sol[0]))) for d in (0, 1)])   # T-2, then T-3
Mg = np.linspace(0.05, m.Mbar, 1000)                          # dense grid for the plots
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for s, lab in ((sol[2], 'last period'), (sol[1], 'one before last')):
    for d, name in ((0, 'retire'), (1, 'work')):
        ax[0].plot(Mg, value(m, s[d], d, Mg), label=f'{lab}, {name}')
vW, vR = value(m, sol[1][1], 1, Mg), value(m, sol[1][0], 0, Mg)
c = np.where(vW > vR, policy(m, sol[1][1], Mg), policy(m, sol[1][0], Mg))   # the worker's consumption
ax[1].plot(Mg, c, label='worker: retire or work')
ax[1].plot(Mg, policy(m, sol[1][0], Mg), label='retiree')
ax[0].set(ylim=(-3, 5), xlabel='wealth M', title='Choice-specific value functions of a worker')
ax[1].set(ylim=(0, 10.5), xlabel='wealth M', title='Consumption one period before the last')
for a in ax: a.grid(color='0.9'); a.legend()
plt.show()

## Why the Euler equation is not enough

In [ ]:
def euler_residual(m, sol1, M, c):
    '''Euler residual of a worker who keeps working, at wealth M and consumption c, given next period solution'''
    pass                                                  # your code here

In [ ]:
M, c, v, ev0 = egm_step(m, 1, sol[1])                         # raw EGM step of the worker, two periods before last
bend = np.flatnonzero(np.diff(M) <= 0)                        # steps where the endogenous grid bends back
lo, hi = M[bend].min() - 0.3, M[bend].max() + 0.3             # window around the zig-zag
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].plot(M, v, '.-', ms=4, lw=0.7)
ax[1].plot(M, c, '.-', ms=4, lw=0.7)
ax[0].set(xlim=(lo, hi), ylim=(np.interp(lo, M, v) - 0.03, np.interp(hi, M, v) + 0.03),
          xlabel='wealth M', title='Value at the EGM points, in savings order')
ax[1].set(xlim=(lo, hi), ylim=(np.interp(lo, M, c) - 0.3, np.interp(hi, M, c) + 0.3),
          xlabel='wealth M', title='Consumption at the EGM points')
cg = np.linspace(1.6, 2.6, 500)
for Mz in np.linspace(lo + 0.1, hi - 0.1, 5):
    ax[2].plot(cg, euler_residual(m, sol[1], Mz, cg), lw=0.8, label=f'M = {Mz:.2f}')
ax[2].axhline(0, color='k', lw=0.5)
ax[2].set(ylim=(-0.15, 0.15), xlabel='consumption c', title='Euler residuals: several zeros'); ax[2].legend()
for a in ax: a.grid(color='0.9')
plt.show()

## The secondary upper envelope

In [ ]:
def segments(M):
    '''Indices of the maximal monotone runs of M, each run ascending in M, the turning points shared'''
    pass                                                  # your code here

In [ ]:
def envelope_lines(m, d, M, c, v, ev0, runs):
    '''Every run on the wealth points inside its own interval, the envelope, and the line on top at every point'''
    pass                                                  # your code here

In [ ]:
def crossing(m, d, ev0, x, lines, a, b, j):
    '''Kink where line b overtakes line a on [x[j], x[j+1]]: wealth, value, consumption left and right'''
    pass                                                  # your code here

In [ ]:
def upper_envelope(m, d, M, c, v, ev0):
    '''Delete the endogenous points below the upper envelope of the curve (M, v), insert the kinks'''
    pass                                                  # your code here

In [ ]:
Mc, cc, vc = upper_envelope(m, 1, M, c, v, ev0)             # the raw points of the figure above, cleaned
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(M, v, '.-', ms=4, lw=0.7, color='0.6', label='EGM points, in savings order')
ax[0].plot(Mc, vc, 'o-', ms=3, lw=1, color='C3', label='after the upper envelope')
ax[1].plot(M, c, '.-', ms=4, lw=0.7, color='0.6', label='EGM points, in savings order')
ax[1].plot(Mc, cc, 'o-', ms=3, lw=1, color='C3', label='after the upper envelope')
ax[0].set(xlim=(lo, hi), ylim=(np.interp(lo, M, v) - 0.03, np.interp(hi, M, v) + 0.03),
          xlabel='wealth M', title='Value: the upper envelope and the secondary kink')
ax[1].set(xlim=(lo, hi), ylim=(np.interp(lo, M, c) - 0.3, np.interp(hi, M, c) + 0.3),
          xlabel='wealth M', title='Consumption: the jump at the secondary kink')
for a in ax: a.grid(color='0.9'); a.legend()
print(f'{M.size} endogenous points in, {Mc.size} out: {np.sum(~np.isin(Mc, M))} inserted at the kink')
plt.show()

## The full algorithm without taste shocks

In [ ]:
def solve_dcegm(m, callback=None):
    '''Backward induction with DC-EGM: returns sol[t][d], t = 0..T-1, d = 0 retire, 1 work'''
    pass                                                  # your code here

def worker(m, sol_t, M):
    '''Policy of a worker in one period: consumption, the probability to work, and the value'''
    pass                                                  # your code here

In [ ]:
t0 = time.perf_counter()
sol = solve_dcegm(m)
print(f'DC-EGM solved {m.T} periods on {m.ngrid} grid points in {time.perf_counter() - t0:.3f} seconds')
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for t, col in zip(range(m.T - 1), plt.cm.viridis(np.linspace(0, 0.9, m.T - 1))):
    c, P, V = worker(m, sol[t], Mg)
    ax[0].plot(Mg, c, color=col, lw=0.9, label=f't = {t}')
    ax[1].plot(Mg, V, color=col, lw=0.9)
    ax[2].plot(Mg, P, color=col, lw=0.9)
ax[0].set(ylim=(0, 8), xlabel='wealth M', title="Worker's consumption, t = 0 (dark) to T-2 (light)")
ax[1].set(ylim=(-4, 10), xlabel='wealth M', title="Worker's value function")
ax[2].set(xlabel='wealth M', title='Decision to keep working')
ax[0].legend(fontsize=8)
for a in ax: a.grid(color='0.9')
plt.show()

### Check against the analytic solution

In [ ]:
# your code here

## Random income

In [ ]:
t = m.T - 4
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for sy, nq, ls in ((0, 5, '-'), (0.2, 5, '-'), (0.5, 5, '-'), (0.5, 30, ':')):
    ms = retirement(sigma_y=sy, nquad=nq)
    c, P, V = worker(ms, solve_dcegm(ms)[t], Mg)
    ax[0].plot(Mg, c, ls, lw=1, label=f'σ_y = {sy}, {nq} nodes')
    ax[1].plot(Mg, P, ls, lw=1)
ax[0].set(xlim=(2, 9), ylim=(1.3, 2.6), xlabel='wealth M', title=f'Consumption of a worker, t = {t}')
ax[1].set(xlabel='wealth M', title='Decision to keep working')
ax[0].legend()
for a in ax: a.grid(color='0.9')
plt.show()

## Extreme value taste shocks

### The smoothed Euler equation

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
Mz = 0.5 * (lo + hi)                                          # a wealth level inside the zig-zag region
for s in (0, 0.01, 0.05, 0.1):
    ms = retirement(sigma=s)
    ss = solve_dcegm(ms)
    c, P, V = worker(ms, ss[t], Mg)
    ax[0].plot(Mg, policy(ms, ss[t][1], Mg), lw=1, label=f'σ = {s}')   # consumption conditional on working
    ax[1].plot(Mg, P, lw=1)
    ax[2].plot(cg, euler_residual(ms, ss[m.T - 2], Mz, cg), lw=1)
ax[0].set(xlim=(2, 9), ylim=(1.3, 2.6), xlabel='wealth M', title=f'Consumption of a worker who keeps working, t = {t}')
ax[1].set(xlabel='wealth M', title='Probability to keep working')
ax[2].axhline(0, color='k', lw=0.5)
ax[2].set(ylim=(-0.15, 0.15), xlabel='consumption c', title=f'Euler residual at M = {Mz:.2f}, t = T-3')
ax[0].legend()
for a in ax: a.grid(color='0.9')
plt.show()

## Simulation

In [ ]:
def simulate(m, sol, M0, seed=None):
    '''Simulate the life cycle of workers from initial wealth M0 (array): wealth, consumption, work status'''
    pass                                                  # your code here

In [ ]:
ms = retirement(sigma=0.1, sigma_y=0.2, nquad=10)
ss = solve_dcegm(ms)
M0 = np.random.default_rng(1).uniform(2, 20, 1000)            # initial wealth of 1000 workers
Ms, Cs, Ds = simulate(ms, ss, M0, seed=2026)
age = np.where((Ds == 0).any(axis=1), np.argmin(Ds, axis=1), ms.T)   # first period of retirement
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for i in range(15):
    ax[0].plot(Ms[i], lw=0.8, color='C0'); ax[1].plot(Cs[i], lw=0.8, color='C1')
    if age[i] < ms.T:
        ax[0].plot(age[i], Ms[i, age[i]], 'ks', ms=4); ax[1].plot(age[i], Cs[i, age[i]], 'ks', ms=4)
ax[2].hist(age, bins=np.arange(ms.T + 2) - 0.5, color='C2')
ax[0].set(xlabel='period', title='Wealth of 15 simulated workers, square: retirement')
ax[1].set(xlabel='period', title='Consumption')
ax[2].set(xlabel='first period of retirement', title='Distribution of the retirement age, 1000 workers')
for a in ax: a.grid(color='0.9')
plt.show()

## Keane's test

Deterministic model, no taste shocks, no income shocks, $\beta R = 1$. The utility is
additively separable in consumption and work, so the Euler equation holds across the
retirement decision too: on any path where the credit constraint does not bind,
consumption is the same in every period, before and after retirement.

In [ ]:
# your code here